# Visualización de tablas e imágenes

Retomaremos los archivos de las sesiones 2 y 3 para representar edades, comparar proporciones y mostrar imágenes. Usaremos Matplotlib, una biblioteca para crear gráficos en Python.

## Contenido
1. Figura, área de dibujo e histograma.
2. Comparación de grupos con barras.
3. Imágenes de CIFAR-10 y exportación.

Los dos ejercicios están dentro de esta notebook.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_dir = Path.cwd()
if not (project_dir / "pyproject.toml").exists():
    raise FileNotFoundError("Open the notebook from the session project directory")
passengers = pd.read_csv(
    project_dir.parent / "sesion-02-pandas/outputs/passengers_prepared.csv"
)
image_dir = project_dir.parent / "sesion-03-pytorch/outputs"
images = np.load(image_dir / "images.npy", allow_pickle=False)
labels = np.load(image_dir / "labels.npy", allow_pickle=False)
report = json.loads((image_dir / "report.json").read_text())
print(passengers.shape, images.shape)

## 1. Figura e histograma

`plt.subplots()` crea una **Figure** (`fig`), que contiene el gráfico, y un **Axes** (`ax`), el área donde dibujamos. `figsize` expresa el tamaño en pulgadas y `layout="constrained"` ajusta el espacio para títulos y etiquetas.

Un histograma agrupa valores numéricos en intervalos. `bins=10` divide el rango en diez intervalos y la altura cuenta pasajeros. Usamos `Age`, la edad original, y omitimos sus ausentes con `dropna()`. Graficar `age_filled` mezclaría edades registradas con reemplazos.

In [ ]:
ages = passengers["Age"].dropna()
print(f"Recorded ages: {len(ages)}; missing: {passengers['Age'].isna().sum()}")
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.hist(ages, bins=10, color="#245E87", edgecolor="white")
ax.set_title(f"Recorded ages (n={len(ages)})")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Passengers")
plt.show()

### Ejercicio 1 · Cambiar los intervalos

Repite el histograma con `bins=20`, conservando los mismos datos y etiquetas. Describe en dos frases qué detalles se distinguen mejor y por qué cambiar los intervalos no cambia la cantidad de edades disponibles.

In [ ]:
# Write your histogram here.

## 2. Comparación con barras

Las barras comparan categorías. La media de `Survived`, que contiene ceros y unos, es la proporción de supervivientes de cada grupo. Multiplicamos por 100 para mostrar porcentajes y conservamos el tamaño de cada grupo en las etiquetas.

El eje vertical empieza en cero y termina en 100. Estas proporciones describen el archivo, sin demostrar que la clase por sí sola explique la supervivencia.

In [ ]:
summary = passengers.groupby("Pclass")["Survived"].agg(["mean", "size"])
class_names = [f"Class {i} (n={n})" for i, n in summary["size"].items()]
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.bar(class_names, summary["mean"] * 100, color="#245E87")
ax.set_title("Observed survival by passenger class")
ax.set_ylabel("Survived (%)")
ax.set_ylim(0, 100)
plt.show()

### Ejercicio 2 · Edades ausentes por clase

Agrupa por `Pclass` y calcula el porcentaje de valores `True` en `age_missing`. Dibuja barras con las tres clases, título, etiqueta vertical y escala de 0 a 100. Usa la operación de agrupación del ejemplo anterior como guía.

Indica qué clase tiene el mayor porcentaje y por qué ese resultado es distinto de contar únicamente cuántas edades faltan.

In [ ]:
# Calculate the percentages and draw the bars here.

## 3. Imágenes de CIFAR-10

Los archivos guardan imágenes como `(N, C, H, W)` con valores entre 0 y 1. `imshow` espera una imagen RGB como `(H, W, C)`. `transpose(1, 2, 0)` mueve los ejes de una imagen de CHW a HWC; `reshape` no intercambia esos ejes.

`subplots(2, 4)` crea ocho áreas de dibujo. `axes.flat` permite recorrerlas una por una y `enumerate` añade el índice. Ocultamos los ejes y usamos la etiqueta real de cada imagen como título. `interpolation="nearest"` permite observar los píxeles originales.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4), layout="constrained")
for index, ax in enumerate(axes.flat):
    ax.axis("off")
    if index < len(images):
        image = images[index].transpose(1, 2, 0)
        ax.imshow(image, interpolation="nearest")
        ax.set_title(report["classes"][int(labels[index])])
fig.suptitle(f"CIFAR-10: first {min(8, len(images))} exported images")
plt.show()

### Guardar una figura

`fig.savefig` guarda esa figura, aunque ya la hayamos mostrado. `dpi` controla la resolución del PNG. Las imágenes pertenecen a la selección exportada en la sesión 3; sus frecuencias no describen necesariamente todo CIFAR-10.

In [ ]:
output_dir = project_dir / "outputs"
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / "cifar_grid.png", dpi=150)
print(output_dir / "cifar_grid.png")

### De la notebook al proyecto

`main.py` reúne los tres ejemplos y guarda sus figuras. `plt.close(fig)` libera cada figura después de guardarla cuando ejecutamos el programa desde la terminal.

```bash
uv run --locked python main.py
```

Antes de interpretar un gráfico, identifica qué datos incluye, qué representa cada eje y cuál es el denominador de sus porcentajes.

[Documentación oficial y fuentes](./FUENTES.md)